In [53]:
spark

In [57]:
%run /opt/spark-notebooks/silver_base/parsed_data.ipynb


Master DataFrame 'parsed_df' successfully created and listening to Bronze!


In [62]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver_base")

DataFrame[]

In [63]:
from pyspark.sql.functions import col
# 1. Create the database table mapping
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_base.delivery_fact_RT (
        order_id STRING,
        partner_id STRING,
        delivery_type STRING,
        dispatch_time TIMESTAMP,
        actual_delivery_time TIMESTAMP,
        expected_delivery_date DATE,
        sla_days INT,
        delivery_attempts INT,
        source_system STRING,
        bronze_ingest_ts TIMESTAMP
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/base/delivery_fact_RT'
""")

DataFrame[]

In [64]:
silver_delivery_fact_df = parsed_df.select(
    col("data.order.order_id").alias("order_id"),
    col("data.order.delivery.delivery_partner.partner_id").alias("partner_id"),
    col("data.order.delivery.delivery_type").alias("delivery_type"),
    col("data.order.delivery.dispatch_time").cast("timestamp").alias("dispatch_time"),
    col("data.order.delivery.actual_delivery_time").cast("timestamp").alias("actual_delivery_time"),
    col("data.order.delivery.expected_delivery_date").cast("date").alias("expected_delivery_date"),
    col("data.order.delivery.sla_days").cast("int").alias("sla_days"),
    col("data.order.delivery.delivery_attempts").cast("int").alias("delivery_attempts"),
    col("source_system"),
    col("bronze_ingest_ts")
)

In [65]:
delivery_fact_query = (
    silver_delivery_fact_df.writeStream
    .format("delta")
    .queryName("silver_delivery_fact_stream")
    .outputMode("append")
    .option("checkpointLocation", "/opt/spark-data/checkpoints/silver/base/delivery_fact_RT/v1")
    .trigger(processingTime="30 seconds")
    .start("/opt/spark-data/delta/silver/base/delivery_fact_RT")
)
print("Started silver_delivery_fact_RT continuous stream!")

Started silver_delivery_fact_RT continuous stream!


In [ ]:
# spark.sql("SELECT * FROM silver_base.order_event_fact_RT").limit(20).toPandas()


In [68]:
spark.sql("SELECT * FROM silver_base.delivery_fact_RT").limit(20).toPandas()


,order_id,partner_id,delivery_type,dispatch_time,actual_delivery_time,expected_delivery_date,sla_days,delivery_attempts,source_system,bronze_ingest_ts
0,ORD-10001,DP-01,STANDARD,2026-01-03 10:00:00,2026-01-05 14:30:00,2026-01-06,3,1,order_service,2026-03-27 16:27:10.891
1,ORD-10002,DP-03,EXPRESS,2026-01-03 11:00:00,NaT,2026-01-05,2,1,order_service,2026-03-27 16:27:10.891
2,ORD-10003,DP-02,STANDARD,2026-01-03 10:30:00,2026-01-06 09:00:00,2026-01-06,3,2,order_service,2026-03-27 16:27:10.891
3,ORD-10004,DP-01,STANDARD,NaT,NaT,2026-01-07,4,0,order_service,2026-03-27 16:27:10.891
4,ORD-10005,DP-02,EXPRESS,2026-01-03 11:10:00,2026-01-04 12:00:00,2026-01-04,1,1,order_service,2026-03-27 16:27:10.891
